In [53]:
import os
import re
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =====================================================================
# 1. 데이터 파싱 및 추출 함수 (VRAM & 속도 동시 추출)
# =====================================================================
def extract_value(pattern, text):
    """정규표현식 패턴에 매칭되는 첫 번째 그룹을 실수형으로 반환합니다."""
    match = re.search(pattern, text)
    return float(match.group(1)) if match else None

def parse_log_file(file_path):
    """로그 파일을 읽어 16-bit부터 2-bit까지의 VRAM과 속도 데이터를 리스트로 반환합니다."""
    if not os.path.exists(file_path): 
        print(f"[경고] 파일을 찾을 수 없습니다: {file_path}")
        return []
        
    with open(file_path, 'r', encoding='utf-8') as f:
        log_text = f.read()
        
    parsed_data = []
    
    # [1] 16-bit (Base 대조군) 데이터 추출
    base_vram = extract_value(r'순수 모델 적재 VRAM:\s*([\d.]+)\s*GB', log_text)
    base_speed = extract_value(r'\[대조군 측정\] 추론 속도[^:]*:\s*([\d.]+)', log_text)
    parsed_data.append({'Bit': '16-bit', 'VRAM_GB': base_vram, 'Speed_tok_sec': base_speed})
    
    # [2] 8, 4, 3, 2-bit (양자화 모델) 데이터 추출
    for bit in [8, 4, 3, 2]:
        q_vram = extract_value(rf'{bit}-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text)
        q_speed = extract_value(rf'{bit}-bit 추론 속도[^:]*:\s*([\d.]+)', log_text)
        parsed_data.append({'Bit': f'{bit}-bit', 'VRAM_GB': q_vram, 'Speed_tok_sec': q_speed})
        
    return parsed_data

# =====================================================================
# 2. 경로 설정 및 데이터프레임 병합
# =====================================================================
log_paths = {
    'Llama_3.2_1B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\Llama_3.2_1B\master_log_Llama_3.2_1B_Distillation_20260406_031434.log',
    'Qwen2.5_1.5B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\Qwen2.5_1.5B\master_log_Qwen2.5_1.5B_Base_RLHF_20260406_010213.log',
    'TinyLlama_1.1B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\TinyLlama_1.1B\master_log_TinyLlama_1.1B_Base_Scratch_20260405_231336.log'
}

all_records = []

for model_name, path in log_paths.items():
    model_data = parse_log_file(path)
    for record in model_data:
        record['Model'] = model_name  # 모델명 메타데이터 추가
        all_records.append(record)

# 추출된 데이터를 Pandas DataFrame으로 변환 후, 결측치(None) 행 제거
df_bench = pd.DataFrame(all_records)
df_bench = df_bench.dropna(subset=['VRAM_GB', 'Speed_tok_sec'])

# # 파싱된 데이터 터미널 확인용
# print("\n📊 [자동 파싱 완료] 추출된 벤치마크 데이터:")
# print(df_bench.to_string(index=False))

# =====================================================================
# 3. 프레젠테이션용 트레이드오프 시각화
# =====================================================================
if not df_bench.empty:
    # 1. 서브플롯 생성
    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=(
            "<b>[이득] 양자화에 따른 VRAM 점유율 감소 (GiB)</b>", 
            "<b>[트레이드오프] 추론 속도 변화 (Tokens/sec)</b>"
        ),
        horizontal_spacing=0.12
    )

    # 2. 색상 매핑
    color_map = {
        'Llama_3.2_1B': '#5A9BD5',    
        'TinyLlama_1.1B': '#E67E22',   
        'Qwen2.5_1.5B': '#9B59B6'
    }

    # 3. 차트 렌더링 (호버 박스 제거 적용)
    for model in df_bench['Model'].unique():
        df_model = df_bench[df_bench['Model'] == model]
        
        # 좌측: VRAM (GiB)
        fig.add_trace(
            go.Scatter(
                x=df_model['Bit'], y=df_model['VRAM_GB'],
                mode='lines+markers',
                name=model,
                line=dict(color=color_map.get(model, '#000')),
                legendgroup=model,
                hoverinfo='none' # 드래그 시 나타나는 툴팁 박스 비활성화
            ), row=1, col=1
        )
        
        # 우측: 속도 (Tokens/sec)
        fig.add_trace(
            go.Scatter(
                x=df_model['Bit'], y=df_model['Speed_tok_sec'],
                mode='lines+markers',
                name=model,
                line=dict(color=color_map.get(model, '#000')),
                legendgroup=model, showlegend=False,
                hoverinfo='none' # 드래그 시 나타나는 툴팁 박스 비활성화
            ), row=1, col=2
        )

    # 4. 수치 텍스트 배치 및 핀포인트 위치 조정
    def apply_smart_annotations(fig, df, metric_col, col_idx):
        for bit in df['Bit'].unique():
            bit_data = df[df['Bit'] == bit].dropna(subset=[metric_col])
            if bit_data.empty: continue
            
            all_vals = bit_data[metric_col].tolist()
            all_vals.sort(reverse=True)
            
            for _, row in bit_data.iterrows():
                val = row[metric_col]
                
                is_top = (val == all_vals[0])
                is_exception = (metric_col == 'Speed_tok_sec' and abs(val - 11.59) < 0.1)
                
                if is_top or is_exception:
                    y_anchor = 'bottom'
                    y_shift = 12 
                else:
                    y_anchor = 'top'
                    y_shift = -12
                
                # --- 특정 수치 핀포인트 예외 처리 ---
                if metric_col == 'Speed_tok_sec' and bit == '16-bit' and abs(val - 11.17) < 0.01:
                    y_shift = -22  # 11.17만 아래로 조금 더 내림
                
                if metric_col == 'VRAM_GB' and bit == '2-bit' and abs(val - 1.20) < 0.01:
                    y_shift = -5   # 1.20만 마커(점)에 바짝 붙이도록 올림
                
                fig.add_annotation(
                    x=bit, y=val,
                    text=f"<b>{val:.2f}</b>",
                    showarrow=False,
                    font=dict(color='black', size=12), # 모든 수치 색상을 검은색으로 통일
                    yanchor=y_anchor,
                    yshift=y_shift,
                    row=1, col=col_idx
                )

    apply_smart_annotations(fig, df_bench, 'VRAM_GB', 1)
    apply_smart_annotations(fig, df_bench, 'Speed_tok_sec', 2)

    # 5. [수정됨] 병목 강조 박스 및 화살표 간격 최적화
    fig.add_vrect(
        x0="16-bit", x1="8-bit", fillcolor="rgba(150,150,150,0.1)", 
        layer="below", line_width=0, row=1, col=2
    )
    
    # 화살표의 끝(Tip)은 선과 겹치지 않는 빈 공간(y=13.5)을 향하게 하고, 
    # 박스 자체는 우측 상단(ax=60, ay=-50)으로 쭉 빼서 여백을 시원하게 확보합니다.
    fig.add_annotation(
        x=0.7, y=15.5, 
        text="<b>Dequantization 오버헤드</b><br><span style='font-size:11px'>(ALU 병목)</span>", 
        showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="black",
        ax=60, ay=-50, # 박스 위치를 우측 상단으로 여유 있게 이동
        bgcolor="white", bordercolor="black", borderwidth=1, borderpad=5,
        font=dict(size=12, color="black"),
        row=1, col=2
    )

    # 6. 레이아웃, X/Y축 여백 최적화
    fig.update_layout(
        title=dict(
            text="<b>양자화 수준(Bit)에 따른 메모리-연산량 트레이드오프 통합 분석</b>",
            font=dict(size=24), x=0.5, y=0.96
        ),
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5),
        width=1300, height=650, 
        margin=dict(t=160, b=50, l=60, r=60)
    )

    fig.update_xaxes(range=[-0.2, 4.2]) 
    
    vram_max, vram_min = df_bench['VRAM_GB'].max(), df_bench['VRAM_GB'].min()
    speed_max, speed_min = df_bench['Speed_tok_sec'].max(), df_bench['Speed_tok_sec'].min()
    
    fig.update_yaxes(title_text="VRAM (GiB)", showgrid=True, range=[vram_min*0.8, vram_max*1.15], row=1, col=1)
    fig.update_yaxes(title_text="Throughput (Tokens/sec)", showgrid=True, range=[speed_min*0.8, speed_max*1.15], row=1, col=2)

    # 7. 마커 스타일링
    for trace in fig.data:
        if isinstance(trace, go.Scatter):
            trace.update(
                marker=dict(
                    symbol='circle',
                    color='white',
                    size=9, 
                    line=dict(color=trace.line.color, width=2.5)
                ),
                line=dict(width=2.5)
            )

    fig.show()

In [37]:
import os
import pandas as pd
import plotly.express as px

# ==========================================
# 1. 다중 비트 경로 및 환경 설정
# ==========================================
BASE_DIR = r"C:\Users\user\SLM\02_cuda_aligned"
TARGET_BITS = ["GPTQ_8bit", "GPTQ_4bit", "GPTQ_3bit", "GPTQ_2bit"] # 분석할 비트 목록

# ==========================================
# 2. 다중 비트 데이터 로더
# ==========================================
df_list = []

# 비트 목록을 순회하며 데이터 수집
for bit in TARGET_BITS:
    model_paths = {
        "Llama_3.2_1B": rf"{BASE_DIR}\Llama_3.2_1B\Distillation\{bit}\quant_log.csv",
        "TinyLlama_1.1B": rf"{BASE_DIR}\TinyLlama_1.1B\Base_Scratch\{bit}\quant_log.csv",
        "Qwen2.5_1.5B": rf"{BASE_DIR}\Qwen2.5_1.5B\Base_RLHF\{bit}\quant_log.csv"
    }

    for model_name, path in model_paths.items():
        if os.path.exists(path):
            try:
                df_temp = pd.read_csv(path, skipinitialspace=True, encoding='utf-8-sig')
                df_temp.columns = df_temp.columns.str.strip().str.lower()
                
                # 열 밀림 보정
                if 'time' in df_temp.columns and df_temp['time'].isna().all():
                    if df_temp.shape[1] >= 5:
                        df_temp['time'] = df_temp.iloc[:, 4] 
                
                if 'time' not in df_temp.columns:
                    continue
                    
                # 순수 숫자 변환
                df_temp['time'] = pd.to_numeric(
                    df_temp['time'].astype(str).str.replace(r'[^\d.]', '', regex=True), 
                    errors='coerce'
                )
                
                df_valid = df_temp.dropna(subset=['time', 'layer', 'module'])
                
                if len(df_valid) > 0:
                    df_valid = df_valid.copy()
                    df_valid['Model'] = model_name
                    df_valid['Bit'] = bit  # 시각화 구분을 위한 비트 파생 변수 추가
                    df_list.append(df_valid)
                    
            except Exception as e:
                print(f"[{model_name} - {bit}] 로드 중 에러: {e}")

if not df_list:
    raise ValueError("분석할 정상적인 데이터가 없습니다.")

df_all = pd.concat(df_list, ignore_index=True)
print(f"✅ 8, 4, 3, 2비트 데이터 로드 완료! (총 {len(df_all)}행)")


import plotly.express as px

# ==========================================
# [차트 1] 비트별 레이어 양자화 소요 시간 (정상 방향 복구 및 스타일 적용)
# ==========================================
# 1) 데이터 집계 및 정렬
layer_sum_df = df_all.groupby(['Bit', 'Model', 'layer'])['time'].sum().reset_index()
layer_sum_df = layer_sum_df.sort_values(by=['Bit', 'Model', 'layer'])

# 2) 사용자 지정 색상
model_colors = {
    'Llama_3.2_1B': '#5A9BD5',
    'TinyLlama_1.1B': '#E67E22',
    'Qwen2.5_1.5B': '#9B59B6'
}

# 3) Plotly Express Line Chart (수평 모드 해제, X=layer, Y=time)
fig1 = px.line(
    layer_sum_df, 
    x='layer', 
    y='time', 
    color='Model', 
    facet_row='Bit', 
    markers=True,
    # [수정됨] category_orders에 'Model' 순서를 명시적으로 추가하여 Qwen을 맨 아래로 배치
    category_orders={
        "Bit": ["GPTQ_8bit", "GPTQ_4bit", "GPTQ_3bit", "GPTQ_2bit"],
        "Model": ["Llama_3.2_1B", "TinyLlama_1.1B", "Qwen2.5_1.5B"] 
    },
    title='<b>비트별 레이어 양자화 소요 시간 (U-Shape 스파이크)</b>',
    labels={'layer': '레이어 번호', 'time': '누적 소요 시간 (초)'},
    color_discrete_map=model_colors,
    height=1000
)

# 4) [핵심] 시간축(Y축) 범위 및 간격 강제 고정
# 요청하신 90~160 범위와 10단위 간격을 실제 시간축(Y축)에 적용하여 동기화합니다.
fig1.update_yaxes(
    range=[90, 170],   # 시간 90~160 강제 고정
    dtick=10,           # 10단위 간격
    showticklabels=True,
    matches='y',        # 모든 서브플롯의 시간축 완벽 동기화
    title_text="소요 시간 (초)"
)

# 5) X축 (레이어) 설정
fig1.update_xaxes(
    tickmode='linear', 
    dtick=2, 
    showticklabels=True, 
    title_text="레이어 번호",
    matches='x'
)

# 6) 마커 스타일 정밀 제어 (symbol='circle', color='white')
# 흰색 마커가 배경에 묻히지 않도록, 테두리(line)는 각 모델의 선 색상을 상속받게 처리합니다.
for trace in fig1.data:
    trace.update(
        marker=dict(
            symbol='circle',
            color='white',            # 마커 내부는 흰색
            size=8,
            line=dict(
                color=trace.line.color, # 테두리 색상은 기존 선 색상 유지
                width=2.5
            )
        ),
        line=dict(width=2.5)
    )

# 7) 레이아웃 디테일
fig1.update_layout(
    width=1200,  # [추가] 차트의 전체 가로 폭을 900px로 고정 (필요에 따라 800~1000 사이로 조절)
    hovermode="x unified",
    template="plotly_white",
    title_font=dict(size=20),
    margin=dict(t=120, b=40, l=40, r=40), 
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        title_text='' # 범례 타이틀 텍스트 제거로 간결함 유지
    )
)

# 우측 서브플롯 타이틀 정제
fig1.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font=dict(size=14, color="black", weight="bold")))

fig1.show()

✅ 8, 4, 3, 2비트 데이터 로드 완료! (총 1848행)


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd

# ==========================================
# 1. 절대 경로 및 분석 대상 설정
# ==========================================
BASE_DIR = Path(r"C:\Users\user\SLM\02_cuda_aligned")

model_paths = {
    "Llama_3.2_1B": "Distillation",
    "Qwen2.5_1.5B": "Base_RLHF",
    "TinyLlama_1.1B": "Base_Scratch"
}
target_bits = ["GPTQ_2bit", "GPTQ_3bit", "GPTQ_4bit", "GPTQ_8bit"]

# ==========================================
# 2. 데이터 로드 및 CSV 구조 불일치 자동 복구
# ==========================================
all_dataframes = []

for model_name, training_method in model_paths.items():
    for bit in target_bits:
        csv_file_path = BASE_DIR / model_name / training_method / bit / "quant_log.csv"
        
        if csv_file_path.exists():
            try:
                # 첫 번째 줄(잘못된 헤더)을 무시하고 순수 데이터만 먼저 읽어옵니다.
                df = pd.read_csv(csv_file_path, header=None, skiprows=1)
                
                # 데이터 열(Column) 개수에 따라 동적으로 헤더를 올바르게 씌워줍니다.
                if df.shape[1] == 5:
                    df.columns = ['layer', 'module', 'loss', 'damp', 'time']
                elif df.shape[1] == 6:
                    df.columns = ['layer', 'module', 'loss', 'samples', 'damp', 'time']
                else:
                    print(f"[경고] {csv_file_path} 구조 이상 (열 개수: {df.shape[1]}). 스킵합니다.")
                    continue
                
                # time 컬럼 실수형 변환
                df['time'] = pd.to_numeric(df['time'], errors='coerce').fillna(0.0)
                
                # 메타데이터 추가
                df['Model'] = model_name
                df['Bit'] = bit
                all_dataframes.append(df)
                
            except Exception as e:
                print(f"[오류] 데이터 로드 실패 ({csv_file_path}): {e}")

if all_dataframes:
    df_all = pd.concat(all_dataframes, ignore_index=True)
else:
    print("❌ [경고] 병합할 데이터가 없습니다. BASE_DIR 경로를 확인하십시오.")
    exit()

# ==========================================
# 3. 데이터 시각화 (프레젠테이션 발표용 최적화 및 지정 색상 적용)
# ==========================================
TARGET_BIT = "GPTQ_8bit" # 필요에 따라 3bit, 2bit 등으로 변경

df_target = df_all[df_all['Bit'] == TARGET_BIT].copy()
target_modules = ['q_proj', 'k_proj', 'v_proj', 'gate_proj', 'up_proj', 'down_proj']

def extract_base_module(module_name):
    module_str = str(module_name)
    for target in target_modules:
        if target in module_str:
            return target
    return "other"

if 'module' in df_target.columns:
    df_target['base_module'] = df_target['module'].apply(extract_base_module)
    df_target = df_target[df_target['base_module'] != 'other']
    
    module_sum_df = df_target.groupby(['Model', 'base_module'])['time'].sum().reset_index()

    # [수정] 요청하신 색상 팔레트 정의
    fill_colors = {
        'Llama_3.2_1B': '#BDE0FE',   
        'Qwen2.5_1.5B': '#E1C6E9',   
        'TinyLlama_1.1B': '#FFE0B2'  
    }
    
    line_colors = {
        'Llama_3.2_1B': '#90CAF9',   
        'Qwen2.5_1.5B': '#CE93D8',   
        'TinyLlama_1.1B': '#FFB74D'  
    }

    # 1. 기본 그래프 생성 (채우기 색상 적용)
    fig2 = px.bar(
        module_sum_df, 
        x='base_module', 
        y='time', 
        color='Model', 
        barmode='group', 
        text_auto='.1f', 
        category_orders={'base_module': target_modules}, 
        title=f'<b>[{TARGET_BIT}] 모듈별 누적 양자화 시간 분석</b>',
        labels={'base_module': '모듈 유형 (Module Type)', 'time': '전체 누적 소요 시간 (초)'},
        color_discrete_map=fill_colors # 지정한 채우기 색상 매핑
    )

    # [수정] 각 막대에 지정된 진한 테두리 색상 및 두께 적용
    for trace in fig2.data:
        model_name = trace.name
        if model_name in line_colors:
            trace.update(
                marker_line_color=line_colors[model_name],
                marker_line_width=2 # 테두리 두께 (원하는 만큼 조절 가능)
            )

    # 2. 완벽하게 대칭되는 영역 구분선 및 타이틀
    # 중앙 점선
    fig2.add_vline(x=2.5, line_width=1.5, line_dash="dot", line_color="rgba(150, 150, 150, 0.8)")
    
    # 좌측(Attention) 및 우측(MLP) 텍스트를 각 영역의 정중앙 좌표(1과 4)에 배치
    fig2.add_annotation(x=1, y=1.08, yref="paper", text="<b>Attention Block</b>", showarrow=False, font=dict(size=14, color="#555"))
    fig2.add_annotation(x=4, y=1.08, yref="paper", text="<b>MLP Block</b>", showarrow=False, font=dict(size=14, color="#555"))

    # 3. 핵심 인사이트 동적 표기 (박스/화살표 제거, 일체형 텍스트)
    if not module_sum_df.empty and module_sum_df['time'].max() > 0:
        
        # 최대 병목 지점
        max_idx = module_sum_df['time'].idxmax()
        max_row = module_sum_df.loc[max_idx]
        max_model = max_row['Model']               
        max_time = max_row['time']
        
        # 빨간 점선 우측 끝에 병목 정보 결합 (가장 깔끔한 방식)
        fig2.add_hline(
            y=max_time, line_width=1.5, line_dash="dash", line_color="rgba(255, 50, 50, 0.6)",
            annotation_text=f"<b>최대 병목: {max_model} ({max_time:.1f}초)</b>  ",
            annotation_position="top right", 
            annotation_font=dict(color="rgba(220, 50, 50, 1)", size=12)
        )

        # GQA 효율성 표식 (화살표, 박스 없이 허공에 띄움)
        q_avg = module_sum_df[module_sum_df['base_module'] == 'q_proj']['time'].mean()
        k_avg = module_sum_df[module_sum_df['base_module'] == 'k_proj']['time'].mean()
        k_max_time = module_sum_df[module_sum_df['base_module'] == 'k_proj']['time'].max()

        # y_shift를 주어 막대 숫자와 겹치지 않게 위로 살짝 띄움
        if k_avg < (q_avg * 0.95):
            gqa_text = "<b>[GQA 효율성 관찰됨]</b><br>Q 대비 K 연산 시간 감소"
            gqa_color = "#3498db" # 차분한 파란색
        else:
            gqa_text = "<b>[GQA 효율성 미관찰]</b><br>Q와 K 연산 시간 유사"
            gqa_color = "#7f8c8d" # 차분한 회색

        fig2.add_annotation(
            x='k_proj', y=k_max_time, yshift=35, # 막대에서 위로 35px 띄움
            text=gqa_text, showarrow=False, # 화살표 완전 제거
            font=dict(size=12, color=gqa_color)
        )

    # 4. 발표용 레이아웃 최적화
    fig2.update_traces(
        textfont_size=12, textposition="outside", cliponaxis=False        
    )
    
    # y축 최대값을 살짝 넉넉하게 주어 텍스트가 잘리지 않도록 보호
    max_y_limit = module_sum_df['time'].max() * 1.25 if not module_sum_df.empty else 1
    
    fig2.update_layout(
        width=1200,
        yaxis_range=[0, max_y_limit], # Y축 천장 확장
        yaxis_title="전체 누적 소요 시간 (초)",
        template="plotly_white",
        # 범례 가로 배치 및 차트 밖 상단 우측으로 이동
        legend=dict(
            title=None, # '모델 계열' 타이틀 생략하여 공간 절약
            orientation="h", yanchor="bottom", y=1.12, xanchor="right", x=1
        ),
        title_font=dict(size=22),
        margin=dict(t=140, b=40, l=40, r=40), # 상단 마진 대폭 확보
        bargroupgap=0.15 # 막대 그룹 간격을 살짝 더 넓혀 답답함 해소
    )

    fig2.show()
else:
    print("❌ 오류: 데이터가 없습니다.")